# 02 · Data center dashboard
The dashboard is one self-contained HTML page, `outputs/dashboard/dc_dashboard.html`: the map library and the US state outlines are already inside it, so it works offline. This notebook only refreshes the data block inside the page with the latest `dc_unified.csv`.
Default view: operating and under-construction sites, medium or high confidence (Figures 3.1 and 3.2 of the thesis).

In [1]:
import json, re
from datetime import date
import pandas as pd
from utils import CLEAN, DASH

Every site becomes one record for the browser (coordinates rounded to 4 decimals, MW to 1). All 1,630 sites are exported; the page applies the filters.

In [2]:
dc = pd.read_csv(CLEAN / "dc_unified.csv")
cols = {"name": "name", "operator": "operator", "status": "status", "lat": "lat", "lon": "lon", "state": "state",
        "county": "county", "facility_mw": "mw", "mw_source": "mw_source", "operating_year": "year",
        "confidence": "confidence", "sources": "sources", "pjm_zone": "zone"}
records = json.loads(dc[list(cols)].rename(columns=cols).round({"lat": 4, "lon": 4, "mw": 1}).to_json(orient="records"))
print(f"{len(records)} sites")

1630 sites


We replace the data block and the update date inside the page.

In [3]:
page = DASH / "dc_dashboard.html"
html = page.read_text()
html = re.sub(r'(<script id="dc-data" type="application/json">).*?(</script>)',
              lambda m: m.group(1) + json.dumps(records) + m.group(2), html, flags=re.S)
html = re.sub(r'(<span id="gen-date">).*?(</span>)', lambda m: m.group(1) + date.today().isoformat() + m.group(2), html)
page.write_text(html)
print("Updated", page.name)

Updated dc_dashboard.html
